# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print metadata
print(f"Title: {metadata.name}\n\nDescription: {metadata.description}\n\nPublished: {getattr(metadata, 'datePublished', None)}\n\nVersion: {getattr(metadata, 'version', None)}\n\nLicense: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets and corresponding field `@id`s.

We access the record sets defined in the metadata and list their identifiers (`@id`), names, and available fields (columns).
All exploration is referenced using `@id` for entity disambiguation.

In [ ]:
# List all record sets and their fields (using @id)
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
    print(f"Total record sets: {len(record_sets)}\n")
    all_rs_ids = []
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        print(f"Record set @id: {rs_id}")
        all_rs_ids.append(rs_id)
        # Try to print fields/columns
        if isinstance(rs, dict):
            if 'field' in rs:
                print("  Fields:")
                fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
                for f in fields:
                    field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
                    print(f"    - {field_id}")
            elif 'column' in rs:
                print("  Columns:")
                columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
                for col in columns:
                    col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
                    print(f"    - {col_id}")
        print()
else:
    print("No record sets found in dataset metadata.")

# If record sets are not at root, try to find from distribution
if not hasattr(metadata, 'recordSet') or not metadata.recordSet:
    if hasattr(metadata, 'distribution'):
        print("Found distributions. Attempting to find record sets from each distribution:")
        for dist in metadata.distribution:
            if hasattr(dist, 'recordSet') and dist.recordSet:
                rs = dist.recordSet
                rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
                print(f"Distribution {getattr(dist, '@id', None)} contains record set: {rs_id}")
                if isinstance(rs, dict) and 'field' in rs:
                    print("  Fields:")
                    fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
                    for f in fields:
                        field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
                        print(f"    - {field_id}")
                elif isinstance(rs, dict) and 'column' in rs:
                    print("  Columns:")
                    columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
                    for col in columns:
                        col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
                        print(f"    - {col_id}")
    else:
        print("No distributions found in dataset for record set discovery.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**You may need to update the record set `@id` values as needed.**

In [ ]:
# Extract data from the discovered record sets
#
# This dataset does not define record sets directly at the root.
# However, it's common for MLCommons Journal and Senscience data to define a main table as the primary record set under distribution.
# We'll use the actual first available record set discovered, or add the value if known.
{
    'comment': 'FAIR2 dataset record set discovery: update the record_set_id variable below as dictated by the previous code cell.'
}

# Example: Assume record_set_id is '@id' of first main table
record_set_id = None
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    rs = metadata.recordSet[0]
    if isinstance(rs, dict) and '@id' in rs:
        record_set_id = rs['@id']
    else:
        record_set_id = rs
elif hasattr(metadata, 'distribution'):
    # Try to find a record set within distribution
    for dist in metadata.distribution:
        if hasattr(dist, 'recordSet') and dist.recordSet:
            rs = dist.recordSet
            if isinstance(rs, dict) and '@id' in rs:
                record_set_id = rs['@id']
            else:
                record_set_id = rs
            break

if not record_set_id:
    print("Could not automatically find main record set `@id`. Please provide it explicitly.")
else:
    print(f"Using record set @id: {record_set_id}")

# List records for this record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {df.shape[0]} records and {df.shape[1]} fields for record set {record_set_id}.")
print("Columns:", list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations such as removing outliers, transforming distributions, and grouping data by key attributes.

All field/column references MUST use the column `@id` exactly as they appeared in previous outputs.

In [ ]:
# Choose a numeric field for demonstration.
# For proteomics datasets, molecular weights, coverage %, or peptide counts are common numeric columns.
# EXAMPLES (replace with actual column IDs as printed above if present):

import numpy as np

# Try the most likely column names present based on mass spec data conventions.
possible_numeric_fields = [
    'MW', # molecular weight
    'Coverage_Pct', # coverage percent
    'Unique_Peptides', # number of unique peptides
    'Abundance', # normalized abundance
]
numeric_field = None
for col in possible_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break

if numeric_field is None:
    print("No known numeric field found in columns. Please set `numeric_field` to a valid numeric column name as discovered above.")
else:
    print(f"Using numeric field: {numeric_field}\n")
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())
    
    # Pick a likely group-by field. Try 'Description' or 'Protein_Group' or similar.
    possible_group_fields = ['Description', 'Protein_Group', 'Protein_Class']
    group_field = None
    for col in possible_group_fields:
        if col in filtered_df.columns:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found; skipping grouping.")

## 5. Visualization
Visualize distribution of the chosen numeric field and, if possible, its relationship to groups or categories.

Plots require matplotlib or seaborn. Columns need to exist (adjust `numeric_field` and `group_field` as appropriate).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(12, 4))
        top_groups = df[group_field].value_counts().nlargest(10).index
        plot_data = df[df[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field, data=plot_data)
        plt.title(f"{numeric_field} by {group_field} (Top 10 Groups)")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've loaded the mass spectrometry dataset, explored record sets, extracted a main data table, and performed basic analysis on protein attributes such as abundance or molecular weight. Filtering, normalization, and grouping operations highlight how to reference fields and record sets by `@id`, ensuring reproducibility and clear traceability.

**To extend this notebook:**
- Explore more fields, especially text-based attributes.
- Perform advanced entity normalization (e.g., by accession or description).
- Analyze differential abundance/counts between experimental conditions if available.
- Export processed DataFrames as FAIR-aligned tabular datasets if needed.